# First Look at pm-edge Captured Data

Goals: verify data quality on the local archive, get a feel for the universe, look for obvious patterns or oddities at multiple timescales.

In [ ]:
# ruff: noqa: F401, I001
import os

os.environ.setdefault(
    "PM_EDGE_LOCAL_FORWARD_INDEX_DIR",
    "/Users/larrymitchell/pm-edge-data/forward_index",
)

from datetime import datetime, timedelta

import pandas as pd

from notebooks.lib.queries import (
    book_state_quality,
    get_connection,
    hourly_snapshot_volume,
    market_movement,
    multi_timescale_aggregation,
    source_breakdown,
)
from notebooks.lib.plots import (
    plot_book_top_over_time,
    plot_hourly_volume,
    plot_movement_distribution,
    plot_multi_timescale_overlay,
    plot_spread_distribution,
)

con = get_connection()

## Section 1: Capture health overview

In [ ]:
df = hourly_snapshot_volume(con)
plot_hourly_volume(df)

In [ ]:
source_breakdown(con)

## Section 2: Data quality verification

In [ ]:
book_state_quality(con, venue="polymarket")

In [ ]:
book_state_quality(con, venue="kalshi")

## Section 3: Market movement

In [ ]:
movement = market_movement(con)
plot_movement_distribution(movement)

A market with no movement has repeated snapshots but no observed change in `top_bid` or `top_ask`. That can mean a stable liquid market, a stale market, or a capture path that is not receiving live updates. Inspect source breakdown and book depth before interpreting it as market behavior.

## Section 4: Pick a market and look at it

In [ ]:
# Pick the market with the most top-of-book changes
most_active = movement.sort_values("distinct_top_bids", ascending=False).iloc[0]
market_id = most_active["market_id"]
print(
    f"Most active market: {market_id} (venue={most_active['venue']}, snapshots={most_active['snapshots']}, distinct_top_bids={most_active['distinct_top_bids']})"
)

aggs = multi_timescale_aggregation(con, market_id)
plot_multi_timescale_overlay(aggs)

## Section 5: Live-watch active Kalshi Eurovision markets

This section watches the most actively-tracked Kalshi Eurovision ranking markets evolve over time. Eurovision 2026 resolves during the broadcast on Saturday evening (May 16). Run this cell repeatedly through the day — after each rsync run, the local archive will have more recent data and the chart will update accordingly.

To refresh the local archive manually before re-running this cell:

```bash
bash ~/ML/pm-edge/deploy/forward_indexer/rsync_to_laptop.sh
```

In [ ]:
# Discover all Kalshi Eurovision markets currently in the captured data
eurovision_markets = con.sql("""
    SELECT market_id, COUNT(*) AS snapshots, MIN(timestamp_utc) AS first_seen
    FROM order_book_snapshots
    WHERE venue = 'kalshi' AND market_id LIKE 'KXEUROVISIONRANK%'
    GROUP BY market_id
    ORDER BY snapshots DESC
""").fetchdf()
eurovision_markets

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(eurovision_markets) == 0:
    print("No Kalshi Eurovision markets found in the captured data. Nothing to plot.")
else:
    market_ids = eurovision_markets["market_id"].tolist()

    fig = make_subplots(
        rows=len(market_ids),
        cols=1,
        subplot_titles=market_ids,
        shared_xaxes=True,
        vertical_spacing=0.02,
    )

    for i, market_id in enumerate(market_ids, start=1):
        df = con.sql(f"""
            SELECT date_trunc('minute', timestamp_utc) AS minute,
                   AVG(top_bid) AS top_bid,
                   AVG(top_ask) AS top_ask
            FROM order_book_snapshots
            WHERE venue = 'kalshi' AND market_id = '{market_id}'
            GROUP BY minute
            ORDER BY minute
        """).fetchdf()

        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_bid"],
                name="top_bid",
                line={"color": "blue"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df["minute"],
                y=df["top_ask"],
                name="top_ask",
                line={"color": "red"},
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )

    fig.update_layout(
        height=200 * len(market_ids),
        title="Eurovision rank markets — top_bid (blue) and top_ask (red) over time",
        margin={"t": 50, "b": 30, "l": 50, "r": 30},
    )
    fig.show()

## Findings

_Fill this in after running the notebook._

### Eurovision watch notes

_Re-run Section 5 throughout the day on May 16 to observe how prediction-market top_bid and top_ask evolve as the Eurovision final approaches and resolves. Note: prices below 0.5 mean the market thinks the country is unlikely to make top 5/10; prices climbing toward 1.0 indicate growing confidence. Markets should converge to either 0 or 1 by resolution._